# AWS Glue Studio Notebook
##### You are now running an AWS Glue Studio notebook on Glue 6.0; to start using your notebook you need to start an AWS Glue Interactive Session.


#### Run this cell to set up and start your Glue 6.0 interactive session.


In [ ]:
%idle_timeout 30
%glue_version 6.0
%worker_type G.1X
%number_of_workers 5


Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5


In [ ]:
# Apache Sedona is configured in the next cell before the Glue session starts.


Extra jars to be included:
s3://pske-prd-customerexperienceadhoc/maintenance/gae_lib/geoanalytics_2.12-2.0.0.jar
s3://pske-prd-customerexperienceadhoc/maintenance/gae_lib/geoanalytics-natives_2.12-2.0.0.jar
Extra py files to be included:
s3://pske-prd-customerexperienceadhoc/maintenance/gae_lib/geoanalytics-2.0.0.zip
s3://pske-prd-customerexperienceadhoc/maintenance/gae_lib/geoanalytics_2.12-2.0.0.jar,s3://pske-prd-customerexperienceadhoc/maintenance/gae_lib/geoanalytics-natives_2.12-2.0.0.jar


In [ ]:
%%configure
{
  "--datalake-formats": "iceberg",
  "--additional-python-modules": "apache-sedona==1.7.2,duckdb,pyarrow,shapely",
  "--conf": "spark.jars.packages=org.apache.sedona:sedona-spark-shaded-3.5_2.12:1.7.2"
}


The following configurations have been updated: {'--datalake-formats': 'iceberg', '--additional-python-modules': 'duckdb, pyarrow, shapely', '--conf': 'spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin --conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.spark_catalog=org.apache.iceberg.spark.SparkSessionCatalog --conf spark.sql.catalog.spark_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog --conf spark.sql.catalog.spark_catalog.warehouse=s3://pske-prd-customerexperienceadhoc/spatial_analysis/'}


In [ ]:
%timeout 280
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import expr
from awsglue import DynamicFrame
from pyspark.sql.functions import col, to_timestamp
import pyspark.sql.functions as F
from sedona.spark import SedonaContext

# Initialize the Glue and Apache Sedona sessions.
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = SedonaContext.create(SparkSession.builder.getOrCreate())
job = Job(glueContext)


Current timeout is None minutes.
timeout has been set to 280 minutes.
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 30
Timeout: 280
Session ID: 0bc276df-3e50-45d4-8b66-715d846dd42e
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--datalake-formats iceberg
--additional-python-modules duckdb, pyarrow, shapely
--conf spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin --conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.spark_catalog=org.apache.iceberg.spark.SparkSessionCatalog --conf spark.sql.catalog.spark_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog --conf spark.sql.catalog.spark_catalog.warehouse=s3://pske-prd-customerexperienceadhoc/spatial_analysis/


In [ ]:
# Apache Sedona is initialized in the previous cell. No proprietary spatial license is required.


In [3]:
# global state Filter
STATE = "DC"
STATE_LOWER = STATE.lower()
print(f"Global state filter set to: {STATE}")

Global state filter set to: DC


In [6]:
#Read all dnb Data
mu_df = spark.sql("""
SELECT 
    /*+ BROADCAST(P, R) */
    -- Company Identification & Linkage
    A.duns_number,
    A.penske_category,
    A.business_name,
    A.tradestyle_name,
    A.line_of_business,
    A.dot_linkage,
    A.formatted_dot_linkage,
    A.us_1987_sic_1,
    A.naics,

    -- Location & Address Fields
    A.street_address,
    A.city_name,
    A.state_province_abbr,
    A.postal_code,
    A.county_name,
    A.latitude,
    A.longitude,

    -- Metrics & Financials
    A.employees_total,
    A.employees_here,
    A.sales_volume_us_dollars,
    A.telephone_number,

    -- Executive & Leadership Contact Information
    A.chief_exec_officer_full_name,
    A.chief_exec_officer_title,
    A.first_executive_first_name,
    A.first_executive_last_name,
    A.first_executive_title,

    -- Refined 7-Tier Sales Target Segment Classification
    CASE 
        WHEN coalesce(A.sales_volume_us_dollars, 0) >= 50000000 THEN '1. Mega ($50M+)'
        WHEN coalesce(A.sales_volume_us_dollars, 0) >= 10000000 THEN '2. Large ($10M - $50M)'
        WHEN coalesce(A.sales_volume_us_dollars, 0) >= 5000000  THEN '3. Upper Mid ($5M - $10M)'
        WHEN coalesce(A.sales_volume_us_dollars, 0) >= 2500000  THEN '4. Lower Mid ($2.5M - $5M)'
        WHEN coalesce(A.sales_volume_us_dollars, 0) >= 1000000  THEN '5. Small ($1M - $2.5M)'
        WHEN coalesce(A.sales_volume_us_dollars, 0) >= 250000   THEN '6. Micro ($250K - $1M)'
        WHEN A.sales_volume_us_dollars IS NULL THEN '8. Unknown (No Data)'
        ELSE '7. Nano / Pre-Rev (<$250K)'
    END AS sales_target_segment,

    -- Polk Vehicle Registration Enrichment
    P.confidence_code AS polk_confidence_code,
    P.total_fleet_size_gvw_3_8,

    -- RigDig Enrichment
    R.confidence_code AS rigdig_confidence_code,
    R.ent_usdot_total_pwr,
    R.eqt_class_3to8_units,
    R.eqt_class_all_units

FROM ptl_marketuniverse.mu_dnb_data_master A
LEFT JOIN ptl_marketuniverse.mu_polk P
    ON A.duns_number = P.duns_number
LEFT JOIN ptl_marketuniverse.mu_rigdig R
    ON A.duns_number = R.duns_number
-- WHERE A.state_province_abbr = 'DC' 
-- ORDER BY A.sales_volume_us_dollars DESC
""")
df_rigdig = spark.read.table("ptl_marketuniverse.mu_rigdig")
print("iceberg table format is read")
df_rigdig = spark.read.table("ptl_marketuniverse.mu_dnb_data_master")
print("Delta format is read")

iceberg table format is read
Delta format is read


In [7]:
# DC filter to 4K and MU total is 14 million
dc_mu_df = mu_df.filter(mu_df.state_province_abbr == STATE)
print(f"Total Market Universe Companies ({STATE}): {dc_mu_df.count()}")
print("\nRefined Breakdown by Sales Target Segment:")
dc_mu_df.groupBy("sales_target_segment") \
    .count() \
    .orderBy("sales_target_segment") \
    .show(truncate=False)

Total Market Universe Companies (DC): 24097

Refined Breakdown by Sales Target Segment:
+--------------------------+-----+
|sales_target_segment      |count|
+--------------------------+-----+
|1. Mega ($50M+)           |591  |
|2. Large ($10M - $50M)    |973  |
|3. Upper Mid ($5M - $10M) |1039 |
|4. Lower Mid ($2.5M - $5M)|805  |
|5. Small ($1M - $2.5M)    |1287 |
|6. Micro ($250K - $1M)    |2565 |
|7. Nano / Pre-Rev (<$250K)|16837|
+--------------------------+-----+


In [8]:
# Read LSR report from Territory analysis
dc_prospects_df = spark.read.format("csv")\
                            .option("header", "true")\
                            .option("inferSchema", "true")\
                            .load("s3://pske-prd-customerexperienceadhoc/spatial_analysis/lsr_imports/LSR_DC_SF_Data.csv")

dc_prospects_df =  dc_prospects_df.toDF(*[col.lower() for col in dc_prospects_df.columns])
dc_prospects_df = dc_prospects_df.drop("BUSINESS_NAME")
dc_prospects_df = dc_prospects_df.drop("STREET_ADDRESS")
print(dc_prospects_df.count())
dc_prospects_df.groupBy("sf_account_id").count().show(truncate=False)

56498
+-------------+-----+
|sf_account_id|count|
+-------------+-----+
|Yes          |2273 |
|No           |54224|
|FIELD        |1    |
+-------------+-----+


In [9]:
# Filter out the sf_account_id (sales force account not present)
dc_ps_join = mu_df.join(dc_prospects_df,"duns_number","inner").where(col("sf_account_id") == "No")
dc_ps_join =  dc_ps_join.filter(col("state") == STATE)
print(f"{STATE} prospect count: {dc_ps_join.count()}")
dc_ps_join.groupBy("sales_target_segment").count().show(truncate=False)

DC prospect count: 4337
+--------------------------+-----+
|sales_target_segment      |count|
+--------------------------+-----+
|6. Micro ($250K - $1M)    |1221 |
|4. Lower Mid ($2.5M - $5M)|189  |
|7. Nano / Pre-Rev (<$250K)|2318 |
|3. Upper Mid ($5M - $10M) |133  |
|5. Small ($1M - $2.5M)    |345  |
|2. Large ($10M - $50M)    |131  |
+--------------------------+-----+


In [10]:
dc_ps_join.printSchema()
dc_ps_join.show(1)

root
 |-- duns_number: string (nullable = true)
 |-- penske_category: string (nullable = true)
 |-- business_name: string (nullable = true)
 |-- tradestyle_name: string (nullable = true)
 |-- line_of_business: string (nullable = true)
 |-- dot_linkage: string (nullable = true)
 |-- formatted_dot_linkage: string (nullable = true)
 |-- us_1987_sic_1: string (nullable = true)
 |-- naics: string (nullable = true)
 |-- street_address: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- state_province_abbr: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- county_name: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- employees_total: long (nullable = true)
 |-- employees_here: long (nullable = true)
 |-- sales_volume_us_dollars: long (nullable = true)
 |-- telephone_number: string (nullable = true)
 |-- chief_exec_officer_full_name: string (nullable = true)
 |-- chief_exec_officer_title: s

In [10]:
# select parcel and filter zoing_typ="Residential"
parcel_src = spark.read.parquet("s3://pske-prd-customerexperienceadhoc/Regrid_Parcels/State_Level_US_Parcels/DC/")
print(f" Total DC counts {parcel_src.count()}")
print(f""" Non Residential DC counts {parcel_src.filter("zoning_typ != 'Residential'").count()}""")
print(f""" Landuse code not residential {parcel_src.filter(F.col("lbcs_activ").isNull() |
    ~(
        (F.col("lbcs_activ").cast("int").between(1000, 1999)) |
        (F.col("lbcs_act_1").ilike("%Household%")) 
    )).count()}""")
# parcel_tx_repartitioned = parcel_tx.repartition(200)

#parcel_tx.select(length(col("geometry"))).alias("wkb_len").describe().show()
#parcel_tx_geom.printSchema()


 Total DC counts 207339
 Non Residential DC counts 34068
 Landuse code not residential 34361


In [11]:
df_zone_nonres = parcel_src.filter(F.col("zoning_typ").isNull() | (F.col("zoning_typ") != "Residential"))
df_lbcs_nonres = parcel_src.filter(F.col("zoning_typ").isNull() | ~ (F.col("lbcs_activ").cast("int").between(1000, 1999)))
parcel_nonres = df_zone_nonres.unionByName(df_lbcs_nonres).dropDuplicates(["parcelnu_1"])
parcel_nonres.count()

46342


In [48]:
anti_parcels = parcel_src.join(parcel_nonres, "parcelnu_1", "left_anti")
anti_parcels.count()
anti_parcels.groupBy("lbcs_activ", "zoning_typ").count().show()

+----------+-----------+------+
|lbcs_activ| zoning_typ| count|
+----------+-----------+------+
|      NULL|Residential|  8343|
|    1100.0|Residential|152519|
|    1200.0|Residential|    60|
|    1300.0|Residential|    54|
+----------+-----------+------+


In [51]:
parcel_nonres.groupBy("lbcs_activ", "zoning_typ").count().show()
parcel_nonres.select("parcelnu_1","usecode","zoning","zoning_des","zoning_typ","zoning_sub","lbcs_activ").show(10, truncate=False)

+----------+-----------+-----+
|lbcs_activ| zoning_typ|count|
+----------+-----------+-----+
|    3000.0|    Special|   22|
|    1100.0|    Special|  823|
|    4700.0|    Special|   54|
|    3120.0|    Special|  307|
|    1100.0|      Mixed|12704|
|      NULL| Commercial|  518|
|    4000.0| Commercial|   22|
|    1300.0| Commercial|    4|
|    6700.0|      Mixed|    7|
|    5210.0| Commercial| 1090|
|    2000.0|      Mixed|  149|
|    2000.0| Commercial|    7|
|    2000.0|    Special|   80|
|    5210.0|      Mixed| 2279|
|    6300.0|Residential|    1|
|    1300.0|      Mixed|    8|
|    2110.0|      Mixed|   44|
|    4100.0|    Special|   18|
|    7240.0|      Mixed|    7|
|    4100.0|Residential|  322|
+----------+-----------+-----+
only showing top 20 rows

+----------+-------+------+----------+----------+----------+----------+
|parcelnu_1|usecode|zoning|zoning_des|zoning_typ|zoning_sub|lbcs_activ|
+----------+-------+------+----------+----------+----------+----------+
|0004N2004 |01

In [ ]:
# Decode the parcel WKB geometry and assign its WGS 84 spatial reference.
parcel_nonres_geom = parcel_nonres.withColumn(
    "polygeom",
    expr("ST_SetSRID(ST_GeomFromWKB(geometry), 4326)")
)
parcel_nonres_geom = parcel_nonres_geom.select(
    "polygeom", "parcelnu_1", "usecode", "zoning", "zoning_des",
    "zoning_typ", "zoning_sub", "lbcs_activ"
)
print(parcel_nonres_geom.count())
parcel_nonres_geom.show(5)
parcel_nonres_geom.printSchema()


46342
+--------------------+----------+-------+------+----------+----------+----------+----------+
|            polygeom|parcelnu_1|usecode|zoning|zoning_des|zoning_typ|zoning_sub|lbcs_activ|
+--------------------+----------+-------+------+----------+----------+----------+----------+
|{"rings":[[[-77.0...| 0004N2010|    017| MU-10| Mixed Use|     Mixed| Mixed Use|    1100.0|
|{"rings":[[[-77.0...|  00080813|    032|  MU-2| Mixed Use|     Mixed| Mixed Use|    1200.0|
|{"rings":[[[-77.0...|  00132015|    017| MU-10| Mixed Use|     Mixed| Mixed Use|    1100.0|
|{"rings":[[[-77.0...|  00132027|    017| MU-10| Mixed Use|     Mixed| Mixed Use|    1100.0|
|{"rings":[[[-77.0...|  00132081|    017| MU-10| Mixed Use|     Mixed| Mixed Use|    1100.0|
+--------------------+----------+-------+------+----------+----------+----------+----------+
only showing top 5 rows

root
 |-- polygeom: geometry (nullable = true)
 |-- parcelnu_1: string (nullable = true)
 |-- usecode: string (nullable = true)
 |--

In [ ]:
# Create WGS 84 point geometry from each prospect's longitude and latitude.
dnb_df_geom = dc_ps_join.withColumn(
    "pointgeom",
    expr("ST_SetSRID(ST_Point(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE)), 4326)")
)
dnb_df_geom.select("latitude", "longitude", "pointgeom").show(2, truncate=90)


+----------+-----------+------------------------------+
|  latitude|  longitude|                     pointgeom|
+----------+-----------+------------------------------+
|+38.919451|-077.031451|{"x":-77.031451,"y":38.919451}|
|+38.976065|-077.019520| {"x":-77.01952,"y":38.976065}|
+----------+-----------+------------------------------+
only showing top 2 rows


In [ ]:
dnb_df_geom.count()
parcel_nonres_geom.printSchema()
dnb_df_geom.printSchema()


4337


In [ ]:
dnb_df_parcel_intersect = dnb_df_geom.join(
    F.broadcast(parcel_nonres_geom),
    expr("ST_Contains(polygeom, pointgeom)"),
    "left"
)
print("spatial join finished")


spatial join finished


In [23]:
dnb_df_parcel_intersect = dnb_df_parcel_intersect.withColumn("parcel_flag", F.when(F.col("parcelnu_1").isNotNull(),"Y").otherwise("N"))
dnb_df_parcel_intersect.groupBy("parcel_flag").count().show()

+-----------+-----+
|parcel_flag|count|
+-----------+-----+
|          N| 2029|
|          Y|14591|
+-----------+-----+


In [26]:
no_dup_window = Window.partitionBy("duns_number").orderBy(F.col("parcel_flag").desc(),F.col("lbcs_activ"))
no_dup_df = dnb_df_parcel_intersect.withColumn("rn", F.row_number().over(no_dup_window)).filter(F.col("rn")==1).drop("rn")
print(no_dup_df.count())

4196


In [ ]:
#.groupBy("lbcs_activ", "lbcs_act_1") \
#    .agg(
#        F.countDistinct("duns_number").alias("distinct_duns_count"),
#        F.count("ll_uuid").alias("total_matched_parcels")
#    ).orderBy(F.col("distinct_duns_count").desc())
# remaining_distribution.show(50, truncate=False)

In [ ]:
no_dup_df.groupBy("parcel_flag").count().show()

Execution Interrupted. Attempting to cancel the statement (statement_id=30)


In [25]:
no_dup_df = no_dup_df.drop("polygeom")
no_dup_df = no_dup_df.withColumnRenamed("pointgeom", "shape")
no_dup_df.printSchema()

root
 |-- duns_number: string (nullable = true)
 |-- penske_category: string (nullable = true)
 |-- business_name: string (nullable = true)
 |-- tradestyle_name: string (nullable = true)
 |-- line_of_business: string (nullable = true)
 |-- dot_linkage: string (nullable = true)
 |-- formatted_dot_linkage: string (nullable = true)
 |-- us_1987_sic_1: string (nullable = true)
 |-- naics: string (nullable = true)
 |-- street_address: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- state_province_abbr: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- county_name: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- employees_total: long (nullable = true)
 |-- employees_here: long (nullable = true)
 |-- sales_volume_us_dollars: long (nullable = true)
 |-- telephone_number: string (nullable = true)
 |-- chief_exec_officer_full_name: string (nullable = true)
 |-- chief_exec_officer_title: s

In [ ]:
geoparquet_s3_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/exports/dnb_parcel_match_dc_geopaq"
no_dup_df.coalesce(1).write \
                .format("geoparquet") \
                .mode("overwrite") \
                .option("compression", "snappy") \
                .save(geoparquet_s3_path)
print("successfully Exported")

In [ ]:
# Write out as a single CSV file with headers
csv_export_s3_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/exports/dnb_parcel_match_dc"
no_dup_df.drop("shape").coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("emptyValue", "") \
    .save(csv_export_s3_path)

print("csv exported")

Execution Interrupted. Attempting to cancel the statement (statement_id=101)


In [ ]:
geoparquet_s3_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/dnb_parcel_match_dc"
dc_pqt = spark.read.format("geoparquet").load(geoparquet_s3_path)
print(dc_pqt.count())


AnalysisException: [PATH_NOT_FOUND] Path does not exist: s3://pske-prd-customerexperienceadhoc/spatial_analysis/dnb_parcel_match_dc.


In [ ]:
df_with_geom = dc_pqt

df_with_geom.select("shape").printSchema()


root
 |-- shape: geometry (nullable = true)


In [29]:
def remove_duplicate_columns(df):
    """Removes duplicate column names from a PySpark DataFrame, keeping the first occurrence."""
    cols = []
    for col_name in df.columns:
        if df.columns.count(col_name) > 1:
            # If column name appears multiple times, keep the first one
            if col_name not in [c for c in cols]:
                cols.append(col_name)
        else:
            cols.append(col_name)
    
    # Select columns by positional index to resolve duplicate references
    return df.select(*[df.schema.names[i] for i, name in enumerate(df.columns) if i == df.columns.index(name)])

parcel_land_usedesc_nodups = remove_duplicate_columns(parcel_land_usedesc)


In [88]:
ATHENA_DB = "ptl_customerexp"
ATHENA_TABLE = "dnb_parcel_intersect_dc"

df_athena = final_joined_df.drop("parcel_geometry")
print(df_athena.count())
spark.sql(f"DROP TABLE IF EXISTS {ATHENA_DB}.{ATHENA_TABLE}")

# 3. Create External Table pointing to S3 location
spark.sql(f"""
    CREATE TABLE {ATHENA_DB}.{ATHENA_TABLE}
    USING parquet
    LOCATION '{S3_OUTPUT_PATH}'    
""")

print(f"Wrote Athena table: {ATHENA_DB}.{ATHENA_TABLE}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {ATHENA_DB}.{ATHENA_TABLE}").show()

4196


In [ ]:
'''Ranking Logic Strategy
When a D&B point hits multiple overlapping parcel polygons, rank them by evaluating four attributes in order:

Commercial/Business  (lbcs_activ): D&B records represent commercial entities. Prioritize non-residential/commercial land uses (e.g., 3000.0 Industrial/Commercial) over residential (1100.0 Household) or NULL codes.
Stacked Parcel Neutrality: If a point sits in a condo stack (ll_stack_u is populated), treat non-stacked base parcels (NULL ll_stack_u) with higher priority unless matching unit-level business activity.
Deterministic Tie-Breaking: Use ll_uuid as a final tie-breaker to prevent non-deterministic partition sorts across distributed Spark tasks.
'''